In [34]:
import pandas as pd

from gsm_benchmarker.results_analyser.utils import pandas_to_latex

from results_notebook_setup import load_results, significant_models


In [2]:
N_BOOT = 2000

In [3]:
results_loader = load_results(n_boot=N_BOOT)
full_results = results_loader.full_results

100%|██████████| 2/2 [00:00<00:00,  2.61it/s]


In [4]:
boots = {
    'GLMM1': {},
    'GLMM2': {},
}

for prompt_result in full_results.values():
    boots['GLMM1'][prompt_result.short_label] = prompt_result.mres.bootstrap_glmm1
    boots['GLMM2'][prompt_result.short_label] = prompt_result.mres.bootstrap_glmm2


In [5]:
def combine_prompt_boots(glmm_key, effect):
    return pd.concat({key: value.summary_df.xs(effect, level=1) for key, value in boots[glmm_key].items()}, axis=0, names=['prompt', 'model'])

boots_combined_dict = {
    'GLMM1-is_variant': combine_prompt_boots('GLMM1', 'is_variant'),
    'GLMM2-is_variant': combine_prompt_boots('GLMM2', 'is_variant'),
    'GLMM2-gamma_c': combine_prompt_boots('GLMM2', 'gamma_c')
}


In [6]:
boots_combined = pd.concat(boots_combined_dict, names=['test'])
boots_combined

boot_ci_upper_log  \
test             prompt          model                                         
GLMM1-is_variant GSM             Mathstral-7B-v0.1                 -0.037108   
                                 Meta-Llama-3-8B                   -0.096309   
                                 Meta-Llama-3-8B-Instruct           0.065518   
                                 Mistral-7B-Instruct-v0.1          -0.089163   
                                 Mistral-7B-Instruct-v0.3           0.036228   
...                                                                      ...   
GLMM2-gamma_c    code-structured gemma-2-2b                         0.169949   
                                 gemma-2-9b                         0.152930   
                                 gemma-2b                           0.148986   
                                 gemma-7b-it                        0.115809   
                                 phi-2                              0.193958   

                                                           boot_ci_lower_log  \
test             prompt          model                                         
GLMM1-is_variant GSM             Mathstral-7B-v0.1                 -1.636772   
                                 Meta-Llama-3-8B                   -1.374645   
                                 Meta-Llama-3-8B-Instruct          -1.541850   
                                 Mistral-7B-Instruct-v0.1          -1.278804   
                                 Mistral-7B-Instruct-v0.3          -1.078176   
...                                                                      ...   
GLMM2-gamma_c    code-structured gemma-2-2b                        -0.226863   
                                 gemma-2-9b                        -0.292218   
                                 gemma-2b                          -0.232739   
                                 gemma-7b-it                       -0.428509   
                                 phi-2                             -0.221453   

                                                           boot_se_log  \
test             prompt          model                                   
GLMM1-is_variant GSM             Mathstral-7B-v0.1            0.411317   
                                 Meta-Llama-3-8B              0.319634   
                                 Meta-Llama-3-8B-Instruct     0.401929   
                                 Mistral-7B-Instruct-v0.1     0.300793   
                                 Mistral-7B-Instruct-v0.3     0.283162   
...                                                                ...   
GLMM2-gamma_c    code-structured gemma-2-2b                   0.103469   
                                 gemma-2-9b                   0.113348   
                                 gemma-2b                     0.096256   
                                 gemma-7b-it                  0.140691   
                                 phi-2                        0.105917   

                                                           boot_mean_log  \
test             prompt          model                                     
GLMM1-is_variant GSM             Mathstral-7B-v0.1             -0.770811   
                                 Meta-Llama-3-8B               -0.708519   
                                 Meta-Llama-3-8B-Instruct      -0.728287   
                                 Mistral-7B-Instruct-v0.1      -0.692053   
                                 Mistral-7B-Instruct-v0.3      -0.506964   
...                                                                  ...   
GLMM2-gamma_c    code-structured gemma-2-2b                    -0.020345   
                                 gemma-2-9b                    -0.068651   
                                 gemma-2b                      -0.032754   
                                 gemma-7b-it                   -0.142654   
                                 phi-2                         -0.002168   

                         

In [49]:
def apply_tick(v):
    return r"\ding{51}" if v else r"\ding{56}"

mmm = boots_combined[~boots_combined.agreement][['boot_significant', 'wald_significant', 'wald_nonconvergent']]
mmm = mmm.reset_index()
mmm.insert(0, 'Test', mmm.test.apply(lambda s: f"{s[:4]} {s[4]}"))
mmm.insert(1, 'Effect', mmm.test.apply(lambda s: 'Variant' if s.endswith('variant') else r'$\gamma_c$'))
mmm.drop('test', inplace=True, axis=1)

mmm['boot_significant'] = mmm.boot_significant.apply(apply_tick)
mmm['wald_significant'] = mmm.wald_significant.apply(apply_tick)
mmm['wald_convergent'] = mmm.wald_nonconvergent.apply(lambda s: apply_tick(not s))
mmm.drop('wald_nonconvergent', inplace=True, axis=1)


bad_rows = []
bad_rows_idx = []
for idx, row in mmm.iterrows():
    if row['Test'] == 'GLMM 1' and row['prompt'] == 'GSM':
        continue
    if row['model'] not in significant_models:
        bad_rows.append(row)
        bad_rows_idx.append(idx)

mmm.drop(bad_rows_idx, inplace=True)

mmm = mmm.sort_values(['boot_significant', 'wald_convergent', 'prompt', 'model', 'Test', 'Effect'])

mmm = mmm.rename(
    columns={
        'wald_significant': "Wald",
        'boot_significant': "Bootstrap",
        'wald_convergent': "Convergent",
        'model': 'Model',
        'prompt': 'Prompt'
    }
)

mmm


,Test,Effect,Prompt,Model,Bootstrap,Wald,Convergent
7,GLMM 2,Variant,GSM,Mathstral-7B-v0.1,\ding{51},\ding{56},\ding{51}
4,GLMM 1,Variant,code-simple,Phi-3.5-mini-instruct,\ding{51},\ding{56},\ding{51}
13,GLMM 2,Variant,code-simple,Phi-3.5-mini-instruct,\ding{51},\ding{56},\ding{51}
0,GLMM 1,Variant,GSM,Meta-Llama-3-8B-Instruct,\ding{56},\ding{51},\ding{51}
15,GLMM 2,$\gamma_c$,GSM,gemma-2b,\ding{56},\ding{51},\ding{51}
9,GLMM 2,Variant,GSM,gemma-2b,\ding{56},\ding{51},\ding{51}
16,GLMM 2,$\gamma_c$,NL-simple,Meta-Llama-3-8B,\ding{56},\ding{51},\ding{51}
3,GLMM 1,Variant,code-simple,Meta-Llama-3-8B,\ding{56},\ding{51},\ding{51}
5,GLMM 1,Variant,code-simple,gemma-7b-it,\ding{56},\ding{51},\ding{51}
1,GLMM 1,Variant,GSM,gemma-2-2b,\ding{56},\ding{51},\ding{56}


In [51]:
print(pandas_to_latex(mmm, index=False, caption='Wald-to-bootstrap mismatch cases', label='tab:wald-mismatch', position='H'))

\begin{table}[H]
\caption{Wald-to-bootstrap mismatch cases}
\label{tab:wald-mismatch}
\begin{tabular}{lccccccc}
\toprule
Test & Effect & Prompt & Model & Bootstrap & Wald & Convergent \\
\midrule
GLMM 2 & Variant & GSM & Mathstral-7B-v0.1 & \ding{51} & \ding{56} & \ding{51} \\
GLMM 1 & Variant & code-simple & Phi-3.5-mini-instruct & \ding{51} & \ding{56} & \ding{51} \\
GLMM 2 & Variant & code-simple & Phi-3.5-mini-instruct & \ding{51} & \ding{56} & \ding{51} \\
GLMM 1 & Variant & GSM & Meta-Llama-3-8B-Instruct & \ding{56} & \ding{51} & \ding{51} \\
GLMM 2 & $\gamma_c$ & GSM & gemma-2b & \ding{56} & \ding{51} & \ding{51} \\
GLMM 2 & Variant & GSM & gemma-2b & \ding{56} & \ding{51} & \ding{51} \\
GLMM 2 & $\gamma_c$ & NL-simple & Meta-Llama-3-8B & \ding{56} & \ding{51} & \ding{51} \\
GLMM 1 & Variant & code-simple & Meta-Llama-3-8B & \ding{56} & \ding{51} & \ding{51} \\
GLMM 1 & Variant & code-simple & gemma-7b-it & \ding{56} & \ding{51} & \ding{51} \\
GLMM 1 & Variant & GSM & gemma-2-2b